<div style="text-align: center;">
    <h1 style="color: #FF6347;">Self-Guided Lab: Retrieval-Augmented Generation (RAGs)</h1>
</div>

<div style="text-align: center;">
    <img src="https://media4.giphy.com/media/v1.Y2lkPTc5MGI3NjExZ3FsdzRveTBrenMxM3VnbDMwaTJxN2NnZm50aGFibXk1NzNnY2Q0MCZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/LR5ZBwZHv02lmpVoEU/giphy.gif" alt="NLP Gif" style="width: 300px; height: 150px; object-fit: cover; object-position: center;">
</div>

<h1 style="color: #FF6347;">Data Storage & Retrieval</h1>


<h2 style="color: #FF8C00;">PyPDFLoader</h2>

`PyPDFLoader` is a lightweight Python library designed to streamline the process of loading and parsing PDF documents for text processing tasks. It is particularly useful in Retrieval-Augmented Generation workflows where text extraction from PDFs is required.

- **What Does PyPDFLoader Do?**
  - Extracts text from PDF files, retaining formatting and layout.
  - Simplifies the preprocessing of document-based datasets.
  - Supports efficient and scalable loading of large PDF collections.

- **Key Features:**
  - Compatible with popular NLP libraries and frameworks.
  - Handles multi-page PDFs and embedded images (e.g., OCR-compatible setups).
  - Provides flexible configurations for structured text extraction.

- **Use Cases:**
  - Preparing PDF documents for retrieval-based systems in RAGs.
  - Automating the text extraction pipeline for document analysis.
  - Creating datasets from academic papers, technical manuals, and reports.


In [3]:
! uv pip install langchain langchain_community pypdf
! uv pip install termcolor langchain_openai langchain-huggingface sentence-transformers chromadb langchain_chroma tiktoken openai python-dotenv

Using Python 3.12.12 environment at: /var/home/anne/Documents/_Ironhack/lab-intro-rag/.venv
Resolved 54 packages in 136ms                                        
⠙ Preparing packages... (0/1)                                                   
⠙ Preparing packages... (0/1)-------------------     0 B/323.73 KiB          
⠙ Preparing packages... (0/1)------------------- 14.92 KiB/323.73 KiB        
⠙ Preparing packages... (0/1)------------------- 30.92 KiB/323.73 KiB        
⠙ Preparing packages... (0/1)------------------- 46.92 KiB/323.73 KiB        
⠙ Preparing packages... (0/1)------------------- 62.92 KiB/323.73 KiB        
⠙ Preparing packages... (0/1)------------------- 78.92 KiB/323.73 KiB        
⠙ Preparing packages... (0/1)------------------- 94.92 KiB/323.73 KiB        
⠙ Preparing packages... (0/1)------------------- 110.92 KiB/323.73 KiB       
⠙ Preparing packages... (0/1)------------------- 126.92 KiB/323.73 KiB       
⠙ Preparing packages... (0/1)2m----------------- 142.92

In [4]:
import langchain
print(langchain.__version__)

1.2.10


In [6]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
import warnings
warnings.filterwarnings('ignore')

/var/home/anne/Documents/_Ironhack/lab-intro-rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<h3 style="color: #FF8C00;">Loading the Documents</h3>

In [7]:
# File path for the document

file_path = "../ai-for-everyone.pdf"

<h3 style="color: #FF8C00;">Documents into pages</h3>

The `PyPDFLoader` library allows efficient loading and splitting of PDF documents into smaller, manageable parts for NLP tasks.

This functionality is particularly useful in workflows requiring granular text processing, such as Retrieval-Augmented Generation (RAG).


In [8]:
# Load and split the document
loader = PyPDFLoader(file_path)
pages = loader.load_and_split()
len(pages)

297

<h3 style="color: #FF8C00;">Pages into Chunks</h3>


####  RecursiveCharacterTextSplitter in LangChain

The `RecursiveCharacterTextSplitter` is the **recommended splitter** in LangChain when you want to break down long documents into smaller, semantically meaningful chunks — especially useful in **RAG pipelines**, where clean context chunks lead to better LLM responses.

####  Parameters

| Parameter       | Description                                                                 |
|-----------------|-----------------------------------------------------------------------------|
| `chunk_size`    | The **maximum number of characters** allowed in a chunk (e.g., `1000`).     |
| `chunk_overlap` | The number of **overlapping characters** between consecutive chunks (e.g., `200`). This helps preserve context continuity. |

####  How it works
`RecursiveCharacterTextSplitter` attempts to split the text **intelligently**, trying the following separators in order:
1. Paragraphs (`"\n\n"`)
2. Lines (`"\n"`)
3. Sentences or words (`" "`)
4. Individual characters (as a last resort)

This makes it ideal for handling **natural language documents**, such as PDFs, articles, or long reports, without breaking sentences or paragraphs in awkward ways.



In [9]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = text_splitter.split_documents(pages)

len(chunks)

1096

####  Alternative: CharacterTextSplitter

`CharacterTextSplitter` is a simpler splitter that breaks text into chunks based **purely on character count**, without trying to preserve any natural language structure.

##### Example:
```python
from langchain.text_splitter import CharacterTextSplitter

text_splitter = CharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
````

This method is faster and more predictable but may split text in the middle of a sentence or paragraph, which can hurt performance in downstream tasks like retrieval or QA.

---

#### Comparison Table

| Feature                        | RecursiveCharacterTextSplitter | CharacterTextSplitter     |
| ------------------------------ | ------------------------------ | ------------------------- |
| Structure-aware splitting      |  Yes                          |  No                      |
| Preserves sentence/paragraphs  |  Yes                          |  No                      |
| Risk of splitting mid-sentence |  Minimal                     |  High                   |
| Ideal for RAG/document QA      |  Highly recommended           |  Only if structured text |
| Performance speed              |  Slightly slower             |  Faster                  |

---

#### Recommendation

Use `RecursiveCharacterTextSplitter` for most real-world document processing tasks, especially when building RAG pipelines or working with structured natural language content like PDFs or articles.

## Best Practices for Choosing Chunk Size in RAG

### Best Practices for Chunk Size in RAG

| Factor                      | Recommendation                                                                                                                                                                                          |
| ---------------------------| ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **LLM context limit**       | Choose a chunk size that lets you retrieve multiple chunks **without exceeding the model’s token limit**. For example, GPT-4o supports 128k tokens, but with GPT-3.5 (16k) or GPT-4 (32k), keep it modest. |
| **Chunk size (in characters)** | Typically: **500–1,000 characters** per chunk → ~75–200 tokens. This fits well for retrieval + prompt without context overflow.                                                                           |
| **Chunk size (in tokens)**  | If using token-based splitter (e.g. `TokenTextSplitter`): aim for **100–300 tokens** per chunk.                                                                                                            |
| **Chunk overlap**           | Use **overlap of 10–30%** (e.g., 100–300 characters or ~50 tokens) to preserve context across chunk boundaries and avoid cutting off important ideas mid-sentence.                                        |
| **Document structure**      | Use **`RecursiveCharacterTextSplitter`** to preserve semantic boundaries (paragraphs, sentences) instead of arbitrary cuts.                                                                                |
| **Task type**               | For **question answering**, smaller chunks (~500–800 chars) reduce noise.<br>For **summarization**, slightly larger chunks (~1000–1500) are OK.                                                          |
| **Embedding model**         | Some models (e.g., `text-embedding-3-large`) can handle long input. But still, smaller chunks give **finer-grained retrieval**, which improves relevance.                                                  |
| **Query type**              | If users ask **very specific questions**, small focused chunks are better. For broader queries, bigger chunks might help.                                                                                  |


### Rule of Thumb

| Use Case                 | Chunk Size      | Overlap |
| ------------------------| --------------- | ------- |
| Factual Q&A              | 500–800 chars   | 100–200 |
| Summarization            | 1000–1500 chars | 200–300 |
| Technical documents      | 400–700 chars   | 100–200 |
| Long reports/books       | 800–1200 chars  | 200–300 |
| Small LLMs (≤16k tokens) | ≤800 chars      | 100–200 |


### Avoid

- Chunks >2000 characters: risks context overflow.
- No overlap: may lose key information between chunks.



<h2 style="color: #FF8C00;">Embeddings</h2>

Embeddings transform text into dense vector representations, capturing semantic meaning and contextual relationships. They are essential for efficient document retrieval and similarity analysis.

- **What are OpenAI Embeddings?**
  - Pre-trained embeddings like `text-embedding-3-large` generate high-quality vector representations for text.
  - Encapsulate semantic relationships in the text, enabling robust NLP applications.

- **Key Features of `text-embedding-3-large`:**
  - Large-scale embedding model optimized for accuracy and versatility.
  - Handles diverse NLP tasks, including retrieval, classification, and clustering.
  - Ideal for applications with high-performance requirements.

- **Benefits:**
  - Reduces the need for extensive custom training.
  - Provides state-of-the-art performance in retrieval-augmented systems.
  - Compatible with RAGs to create powerful context-aware models.


In [11]:
from dotenv import load_dotenv

In [12]:
load_dotenv()

True

In [13]:
import os

api_key = os.getenv("OPENAI_API_KEY")
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

<h2 style="color: #FF8C00;">ChromaDB</h2>

ChromaDB is a versatile vector database designed for efficiently storing and retrieving embeddings. It integrates seamlessly with embedding models to enable high-performance similarity search and context-based retrieval.

### Workflow Overview:
- **Step 1:** Generate embeddings using a pre-trained model (e.g., OpenAI's `text-embedding-3-large`).
- **Step 2:** Store the embeddings in ChromaDB for efficient retrieval and similarity calculations.
- **Step 3:** Use the stored embeddings to perform searches, matching, or context-based retrieval.

### Key Features of ChromaDB:
- **Scalability:** Handles large-scale datasets with optimized indexing and search capabilities.
- **Speed:** Provides fast and accurate retrieval of embeddings for real-time applications.
- **Integration:** Supports integration with popular frameworks and libraries for embedding generation.

In [15]:
db = Chroma.from_documents(chunks, embeddings, persist_directory="./chroma_db_LAB")
print("ChromaDB created with document embeddings.")

ChromaDB created with document embeddings.


<h1 style="color: #FF6347;">Retrieving Documents</h1>


### Exercice1: Write a user question that someone might ask about your book’s topic or content.

In [18]:
user_question = "What is the importance of AI today?" # User question
retrieved_docs = db.similarity_search(user_question, k=10) # k is the number of documents to retrieve

In [17]:
# Display top results
for i, doc in enumerate(retrieved_docs[:3]): # Display top 3 results
    print(f"Document {i+1}:\n{doc.page_content[36:1000]}") # Display content

Document 1:
will be needed to address the various challenges with regards to  
the development of artificial intelligence. Which formal method can be used 
to test for algorithmic bias? Can we identify simple to use measures to assess 
bias, similar to the way we assess reliability and validity? What is the best way to 
bridge (deep) learning and privacy? Should learning be conducted on the user 
side (with algorithms requiring new data)? Or should data be transferred to a 
trusted intermediary who performs the analysis on behalf of firms? Do users 
need to be compensated in one way or another for data or resources provided? 
Moreover, how can the refusal to share data lead to biases in the data available 
for learning? Which data sources can and should be used for algorithmic learn-
ing? Are there certain types of data that should be ‘off-limits’? What role will 
interdisciplinary AI teams play in establishing coexistence between humans
Document 2:
0. Contextualizing Platform Labor. C

<h2 style="color: #FF8C00;">Preparing Content for GenAI</h2>

In [19]:
def _get_document_prompt(docs):
    prompt = "\n"
    for doc in docs:
        prompt += "\nContent:\n"
        prompt += doc.page_content + "\n\n"
    return prompt

In [20]:
# Generate a formatted context from the retrieved documents
formatted_context = _get_document_prompt(retrieved_docs)
print("Context formatted for GPT model.")

Context formatted for GPT model.


In [21]:
formatted_context

'\n\nContent:\n162 AI for Everyone?\nToday, thanks to the rise of big data, computing power and mathematical \nadvancements, and the introduction of convolutional neural networks (CNNs), \nwe live with intelligent algorithms (i.e. weak AI), in many aspects of life. 1 For \nexample, the effects of these algorithms in digital visual production covers rec-\nommendation systems, automatic image editing, analysing and even creating \nnew images, but these are not recognised as ‘intelligent’ systems (Manovich \n2020). What fascinates the human mind are still the observances of failures. \nA prominent example is the images and videos created with the Deep Dream \nalgorithm, which was originally devised to unearth what lies in the hidden lay-\ners of CNNs to capture the workings and failures of the system (Simonyan and \nZisserman 2014). These images are hailed by some as artworks on their own \n(Miller 2019).\nAutonomous AI systems such as self-driving cars, or autonomous lethal\n\n\nContent:

<h2 style="color: #FF8C00;">ChatBot Architecture</h2>

### Exercice2: Write a prompt that is relevant and tailored to the content and style of your book.

In [22]:
prompt = f"""
## SYSTEM ROLE
You are a factual chatbot design to answer specific questions about the impacts of AI.
Your answers must be based on the provided content from a reliable source.

## USER QUESTION
The user has asked:
"{user_question}"

## CONTEXT
Here is the content extracted from the reliable source:
'''
{formatted_context}
'''

## GUIDELINES

1. **Accuracy**:
- Only use the content from the 'CONTEXT' section as a source.
- If the answer cannot be obtained from the provided material, explicit state: "I cannot provide a answer based on the current resources."

2. **Transparency**:
- Reference the book's name and page numbers when providing information.
- You are allowed to point out when the text is lacking depth to properly answer the user's question.
- You are allowed to point out possible resources that could provide an answer.
- You are not allowed to speculate about the answer.

3. **Clarity**:
- Use simple, polite and concise language.
- Format your response in Markdown for readability.

## TASK
- Answer the user's question if possible.
- Point the user to the relevant parts of the source materia.
- If a question cannot be answered with the provided material, brainstorm possible places the user can refer to.

## RESPONSE FORMAT
# [Brief Title of the Answer]
[Answer in simple, clear text.]

**Source**:
• [Book Title], Page(s): [...]
"""

print("Prompt constructed.")

Prompt constructed.


In [23]:
print(prompt)


## SYSTEM ROLE
You are a factual chatbot design to answer specific questions about the impacts of AI.
Your answers must be based on the provided content from a reliable source.

## USER QUESTION
The user has asked:
"What is the importance of AI today?"

## CONTEXT
Here is the content extracted from the reliable source:
'''


Content:
162 AI for Everyone?
Today, thanks to the rise of big data, computing power and mathematical 
advancements, and the introduction of convolutional neural networks (CNNs), 
we live with intelligent algorithms (i.e. weak AI), in many aspects of life. 1 For 
example, the effects of these algorithms in digital visual production covers rec-
ommendation systems, automatic image editing, analysing and even creating 
new images, but these are not recognised as ‘intelligent’ systems (Manovich 
2020). What fascinates the human mind are still the observances of failures. 
A prominent example is the images and videos created with the Deep Dream 
algorithm, which was o

In [24]:
import openai

### Exercice3: Tune parameters like temperature, and penalties to control how creative, focused, or varied the model's responses are.

In [25]:
# Set up GPT client and parameters
client = openai.OpenAI()
model_params = {
    'model': 'gpt-4o',
    'temperature': 0.9,  # Increase creativity
    'max_tokens': 3000,  # Allow for longer responses
    'top_p': 0.9,        # Use nucleus sampling
    'frequency_penalty': 0.5,  # Reduce repetition
    'presence_penalty': 0.6   # Encourage new topics
}

<h1 style="color: #FF6347;">Response</h1>


In [54]:
messages = [{'role': 'user', 'content': prompt}]
completion = client.chat.completions.create(messages=messages, **model_params, timeout=120)

In [55]:
answer = completion.choices[0].message.content
print(answer)

# Importance of AI Today

AI has become a crucial part of modern life due to its ability to enhance various sectors through the use of big data, advanced computing power, and convolutional neural networks (CNNs). It impacts many aspects of our daily lives by enabling intelligent algorithms in areas such as digital visual production, autonomous systems like self-driving cars, language translation, and medical diagnostics. AI also offers potential solutions to global challenges like climate change and poverty, while promoting innovation and economic prosperity.

Moreover, AI is considered one of the most strategic technologies of the 21st century because it can significantly transform society by improving individual well-being and societal progress. AI systems are instrumental in achieving sustainable development goals such as gender balance, climate action, and efficient resource utilization.

**Source**:
• "AI for Everyone?", Page(s): 152-162


<img src="https://miro.medium.com/v2/resize:fit:824/1*GK56xmDIWtNQAD_jnBIt2g.png" alt="NLP Gif" style="width: 500px">

<h2 style="color: #FF6347;">Cosine Similarity</h2>

**Cosine similarity** is a metric used to measure the alignment or similarity between two vectors, calculated as the cosine of the angle between them. It is the **most common metric used in RAG pipelines** for vector retrieval.. It provides a scale from -1 to 1:

- **-1**: Vectors are completely opposite.
- **0**: Vectors are orthogonal (uncorrelated or unrelated).
- **1**: Vectors are identical.


<img src="https://storage.googleapis.com/lds-media/images/cosine-similarity-vectors.original.jpg" alt="NLP Gif" style="width: 700px">

<h2 style="color: #FF6347;">Keyword Highlighting</h2>

Highlighting important keywords helps users quickly understand the relevance of the retrieved text to their query.

In [28]:
from termcolor import colored

The `highlight_keywords` function is designed to highlight specific keywords within a given text. It replaces each keyword in the text with a highlighted version using the `colored` function from the `termcolor` library.


In [29]:
def highlight_keywords(text, keywords):
    for keyword in keywords:
        text = text.replace(keyword, colored(keyword, 'green', attrs=['bold']))
    return text

In [56]:
messages.append({"role": "assistant", "content": answer})
new_question = "create 3 keywords based on your previous answer. The terms should not be composed words. The words should be words that were repeated, and should be present in the last message."
messages.append({"role": "user", "content": new_question})

completion = client.chat.completions.create(messages=messages, **model_params, timeout=120)


In [57]:
import re
raw_keywords = completion.choices[0].message.content
# print(raw_keywords)

query_keywords = re.sub(r'[^a-zA-Z\s]', ' ', raw_keywords)
query_keywords = re.sub(r'\s+', ' ', query_keywords).strip()
query_keywords = query_keywords.split(" ")
query_keywords = query_keywords[:4]
print(query_keywords)


['AI', 'Technology', 'Society']


In [52]:
len(retrieved_docs)

10

### Exercice4: add your keywords

In [58]:
# query_keywords = [] # add your keywords
for i, doc in enumerate(retrieved_docs):
    snippet = doc.page_content[:200]
    highlighted = highlight_keywords(snippet, query_keywords)
    print(f"Snippet {i+1}:\n{highlighted}\n{'-'*80}")

Snippet 1:
162 AI for Everyone?
Today, thanks to the rise of big data, computing power and mathematical 
advancements, and the introduction of convolutional neural networks (CNNs), 
we live with intelligent algo
--------------------------------------------------------------------------------
Snippet 2:
tational power to develop AI systems, are financially strong enough to hire the 
brightest AI talent and have access to the gigantic datasets that are needed to 
train machine learning and deep learni
--------------------------------------------------------------------------------
Snippet 3:
1950s at a series of academic workshops organised at Dartmouth College, New  
Hampshire (USA). A group of scientists, led by mathematics professor John 
McCarthy, gathered to investigate the ways in w
--------------------------------------------------------------------------------
Snippet 4:
Introduction: Why We Need Critical Perspectives on AI  5
potential to surpass human intellectual capacities. T

1. `query_keywords` is a list of keywords to be highlighted.
2. The loop iterates over the first document in retrieved_docs.
3. For each document, a snippet of the first 200 characters is extracted.
4. The highlight_keywords function is called to highlight the keywords in the snippet.
5. The highlighted snippet is printed along with a separator line.

<h1 style="color: #FF6347;">Bonus</h1>

**Try loading one of your own PDF books and go through the steps again to explore how the pipeline works with your content**:
